# Pamięć proceduralna i inżynieria umiejętności

## Cel: Implementacja "mięśni" agenta poprzez modularne umiejętności w Software 3.0

### Czego się nauczysz:
1.  Czym różni się statyczna wiedza (semantic memory) od dynamicznej procedury (procedural memory)?
2.  Jak unikać anty-wzorca "Boga-Agenta" (jeden gigantyczny prompt) poprzez dekompozycję logiki na atomowe Umiejętności (*Skills*)
3.  Dlaczego oddzielamy definicję ("Przepis") od wykonania ("Kucharz") – architektura **Pasywnych Umiejętności**
4.  Jak zbudować profesjonalny kontrakt umiejętności przy użyciu **Pydantic** (Schema) i **Jinja2** (Dynamic Prompts)
5.  Gdzie leży granica inżynierska między *Skillem* a *Sub-Agentem* (i dlaczego na tym etapie wybieramy Skills)
6.  Jak zarządzać stanem w rozmowie wielowątkowej (Multi-turn), czyli skąd Agent wie, że "zadanie jest skończone"
7.  Jak diagnozować błędy desynchronizacji ("LLM zrozumiał, ale Python nie zapisał") na styku modelu i logiki biznesowej

Kojarzysz tę scenę z Matrixa, kiedy Neo podłącza się do symulatora i po kilku sekundach otwiera oczy, mówiąc: "Znam Kung Fu"? Dokładnie to będziemy dzisiaj robić. Do tej pory nasz system miał tożsamość (*Core Memory*) i wiedzę (*Semantic Memory*), ale nie potrafił niczego *zrobić*. Brakowało mu procedur.

Większość początkujących inżynierów AI wpada w pułapkę budowania "Boga-Agenta". Piszą jedną gigantyczną pętlę decyzyjną, która próbuje obsłużyć wszystko naraz: powitanie, diagnostykę, naukę i planowanie. Kończy się to plikiem `main.py` mającym tysiąc linii, pełnym instrukcji warunkowych, którego nikt nie chce dotykać. To klasyczne *spaghetti code*, tylko że zasilane tokenami.

Podejdziemy do tego inaczej. Zamiast uczyć model wszystkiego naraz, stworzymy modularne umiejętności (*Skills*). Dzięki temu nasz system będzie mógł dynamicznie "wgrywać" sobie odpowiednią procedurę w zależności od potrzeby.

## Status prac i przypomnienie z lekcji 2.1

Wróćmy na chwilę do fundamentów. W lekcji 2.1 narysowaliśmy mapę całego terytorium – 7 warstw pamięci systemu Software 3.0. Do tej pory przerobiliśmy 5 z nich.

Sprawdźmy, gdzie dokładnie jesteśmy:
1.  **🏛️ Constitutional Memory** –  ✅ "DNA agenta". Zasady bezpieczeństwa i etyki.
2.  **⚡ Working Memory** –  ✅ Kontekst bieżącej sesji (RAM). To, o czym rozmawiamy "teraz".
3.  **🎯 Workflow Memory** –  ✅ *Secret Sauce*. Maszyna stanów (State Machine). Gdzie jestem w procesie?
4.  **📚 Episodic Memory** –  ✅ Co się wydarzyło? Historia interakcji.
5.  **🧠 Semantic Memory** –  ✅ Co wiem? Baza wiedzy (RAG), fakty o świecie.
6.  **🛠️ Procedural Memory** – 🟢 **Jesteś tutaj** Jak wykonać zadanie? Definicje umiejętności.
7.  **🔧 Resource Memory** – Czy mam zasoby? To temat na **Moduł 3** (narzędzia i świat zewnętrzny).

![](../images/memory_blue_print.jpg)

### Dlaczego zostawiliśmy to na koniec?

To nie przypadek, że **pamięć proceduralna** (*Procedural Memory*) pojawia się dopiero teraz. Mamy już dane (*semantic*), mamy stan (*workflow*), mamy tożsamość (*core*). Ale to nie wystarczy. Pamięć proceduralna to ta iskra, która zamienia dane w konkretne działanie.

### Test roweru

Najlepiej zrozumieć to na przykładzie. Wyobraź sobie, że proszę Cię o napisanie instrukcji: "Jak jeździć na rowerze". Możesz napisać elaborat o fizyce, żyroskopach, kącie nachylenia ciała i sile nacisku na pedały. To jest **wiedza semantyczna**.

Ale jeśli dam ten tekst osobie, która nigdy nie jeździła, i posadzę ją na rowerze – przewróci się po metrze.

Z drugiej strony, Ty po prostu wsiadasz i jedziesz. Nie myślisz o wektorach sił. Twoje ciało po prostu wie, co robić. To jest właśnie **pamięć proceduralna**. To umiejętność, a nie zbiór faktów.

W świecie AI mamy ten sam problem. Model językowy przeczytał cały internet, więc teoretycznie "wie" wszystko o byciu nauczycielem. Ale "wiedzieć" a "przeprowadzić skuteczną lekcję" to dwie zupełnie różne rzeczy.

### Prawdziwe wyzwanie

Tutaj dochodzimy do ściany. Temat pamięci proceduralnej jest jednym z najbardziej dyskutowanych obszarów badawczych. Nie ma jednego standardu. Próba odwzorowania ludzkich procedur w kodzie zawsze będzie pewnym uproszczeniem.

Zamiast tonąć w akademickich dyskusjach, podejdziemy do tego pragmatycznie. Chcę Ci pokazać rozwiązanie, które po wielu iteracjach i błędach okazało się u nas najbardziej stabilne i modularne.

Będziemy nazywać te moduły **Skills** (czyli umiejętności).

Nie ukrywam, że inspiracją był tu ruch, jaki wykonało Anthropic w swoim **Claude Code**. Choć ich implementacja techniczna różni się od naszej, kierunek myślenia jest podobny. Zbudujemy system, w którym umiejętności są wymienne jak kartridże w konsoli – wpinasz "diagnostykę" i agent diagnozuje, wpinasz "planowanie" i agent planuje.

Bez magii. Czysta inżynieria.

## Brakujące ogniwo: pamięć proceduralna

Popatrzmy na warstwy pamięci z tej strony. Mamy już:

```
├── Core Memory        → KIM jest agent (tożsamość)
├── Semantic Memory    → CO agent WIE (fakty)
├── Workflow Memory    → GDZIE jesteśmy (proces)
└── Working Memory     → O CZYM rozmawiamy (kontekst)
```

Logika biznesowa naturalnie przenika przez te warstwy, ale rzeczywistość jest zbyt skomplikowana, by opisać ją w jednym prompcie. Potrzebujemy mechanizmu **dekompozycji**.

Najlepiej zrozumieć to na przykładzie różnicy między encyklopedią a instrukcją obsługi:
- **Semantic Memory:** "Wiem, że w Pythonie do pętli służy `for`." (Fakt)
- **Procedural Memory:** "Wiem, jak zaimplementować sortowanie bąbelkowe." (Procedura)

W systemach agentowych *Skills* to zmodularyzowane kawałki logiki, które agent "uruchamia", by osiągnąć cel.

### Problem kontekstu

Wyobraź sobie, że nasz AI Tutor musi obsłużyć w trakcie jednej sesji cztery skrajnie różne tryby działania:

1.  **Onboarding (Point A)** - Agent działa jak urzędnik. Musi zebrać konkretne 7 pól danych. Kreatywność jest tu niewskazana.
2.  **Goal Setting (Point B)** - Rola doradcy kariery. Wymagana empatia i synteza celów użytkownika.
3.  **Tutoring Mode (Główna pętla)** - "Wolna amerykanka". Konwersacja, poprawianie błędów, tłumaczenie. Priorytetem jest flow.
4.  **Plan Creation** - Czysta analityka. Generowanie ustrukturyzowanego JSON-a na podstawie zebranych danych.

Nie da się napisać jednego promptu, który będzie świetnym urzędnikiem, empatycznym doradcą i kreatywnym nauczycielem jednocześnie. Próba zrobienia tego kończy się halucynacjami. Model gubi się w instrukcjach.

```
┌─────────────────────────────────────────┐
│         AI Tutor System                 │
│   (User: Alice, Workflow: Point A)      │
└─────────────────────────────────────────┘
         │
         ▼
   ❓ Jaki "kapelusz" założyć teraz?
         │
    ┌────┴────┬─────────┬──────────┐
    ▼         ▼         ▼          ▼
  Point A  Point B  Tutoring    Plan
```

Rozwiązaniem nie są lepsze prompty, ale zmiana architektury. Musimy wydzielić te tryby.

## Ewolucja podejścia

Skoro wiemy już, że jeden gigantyczny prompt to ślepa uliczka, jak to zdekomponować? Oszczędzę Ci czasu i opowiem o błędach, które popełniliśmy.

### Ślepa uliczka: Armia Agentów (`Multi-Agent Swarm`)
Plan wydawał się genialny: stwórzmy dziesiątki agentów. Jeden od gramatyki, jeden od powitań, jeden od planowania.

**Co na to rzeczywistość?** Koszmar synchronizacji. Kiedy Agent nr 5 kończył pracę, musiał przekazać cały kontekst do Agenta nr 8. Utrzymanie spójności rozmowy między dziesięcioma bytami wymagało kodu orkiestracji trudniejszego niż samo zadanie. Złożoność połączeń zabiła projekt. Potrzebujemy czegoś lżejszego.

## Architektura: czym właściwie jest *Skill*?

Ustalmy jedną fundamentalną rzecz: **Umiejętność (*Skill*) to tylko przepis.**

Jak mówi filozofia Uniksa:
> "Write programs that do one thing and do it well. Write programs to work together. Write programs to handle text streams, because that is a universal interface."

W naszym systemie oznacza to jasny podział odpowiedzialności:

### Trzy warstwy systemu (analogia restauracji)

```
WARSTWA 1: AgentCoordinator (Manager restauracji)
└── Odpowiedzialność: Decyzja i Stan.
    Nie dotyka jedzenia (danych) ani kuchni (LLM).
    Patrzy na stoliki (State) i decyduje: "Ten klient potrzebuje menu".

WARSTWA 2: TutorAgent (Kelner / Executor)
└── Odpowiedzialność: Wykonanie i Transport.
    Pobiera przepis (Skill), zanosi do Kuchni (LLM) 
    i wraca z gotowym daniem (Response).

WARSTWA 3: Skills (Książka kucharska)
└── Odpowiedzialność: Deklaracja.
    To są czyste przepisy. Pliki te mówią TYLKO: "żeby zrobić danie X,
    potrzebujesz składników Y i powiedz kucharzowi Z".
    Same z siebie nic nie robią.
```

### ☝️ Złota zasada
**Skills NIE GOTUJĄ – one OPISUJĄ, jak gotować.**

W plikach, które będziemy tworzyć w katalogu `skills/`, nie znajdziesz ani jednego wywołania `API call`. To dzieje się tylko w warstwie egzekucyjnej.

## *Skills* jako moduły

Każdy tryb wydzielamy do osobnego, niezależnego pliku.

```python
class PointASkill:
    name = "point_a"
    template = "templates/point_a.j2"  # Prompt template
    output_schema = PointAOutput       # Struktura odpowiedzi (Pydantic)

class TutoringSkill:
    name = "tutoring"
    template = "templates/tutoring.j2"
    output_schema = TutorResponse

# Agent wybiera skill i go wykonuje
class TutorAgent:
    def execute(self, skill_name: str, user_message: str, state: dict) -> dict:
        """
        Flow:
        1. Load skill from registry (discovery)
        2. Prepare execution context (state, history, collected fields)
        3. Skill renders prompt template
        4. Build messages payload (system + history + current message)
        5. ★ CALL LLM API ★ (ONLY PLACE IN ENTIRE SYSTEM!)
        6. Update memory from skill output (via MemoryManager)
        7. Update conversation buffer
        """
        # 1. Skill discovery - load from registry
        skill = self._get_skill(skill_name)

        # 2. Prepare context - gather all info skill needs
        context = self._prepare_context(user_message, state, skill_name)

        # 3. Skill renders prompt (declarative - no execution!)
        prompt = skill.prepare_prompt(context)

        # 4. Build messages - system + optional history + current message
        messages = self._build_messages(prompt, user_message, state, skill_name)

        # 5. ★★★ CALL LLM API ★★★
        # This is the ONLY place in the entire system where we make LLM calls!
        # All retry logic, cost tracking, logging should be added here.
        from config import Config

        response = self.client.chat.completions.create(
            model=Config.MODEL_NAME, #e.g. "gpt-4o-mini"
            messages=messages,
            response_model=skill.output_model
        )

        # 6. Update memory from skill output
        # MemoryManager handles skill-specific mapping (e.g., PointAOutput -> profile)
        new_state = self.memory_manager.update_from_output(
            skill_name=skill_name,
            output=response,
            state=state
        )

        # 7. Update conversation buffer
        # Append both user message and AI response to history
        new_state = self._update_conversation_buffer(
            new_state,
            user_message,
            response.ai_response
        )

        return {
            "ai_response": response.ai_response,
            "new_state": new_state,
        }
```

### Zalety
- ✅ **Modułowy**: każdy *skill* w osobnym pliku.
- ✅ **Testowalny**: możesz testować prompt do onboardingu niezależnie od reszty.
- ✅ **Reużywalny**: Point A może być użyty w innym projekcie.
- ✅ **Czysty**: *Skill* to tylko definicja, nie logika biznesowa.

### A jak możemy uruchomić tę umiejętność?

Skoro *Skill* to tylko przepis, ktoś musi go wykonać. Mamy tu dwie główne ścieżki:

1.  **Przez Głównego Agenta** – Agent "zmienia kapelusz". Ładuje umiejętność do swojej pamięci operacyjnej, wykonuje ją w kontekście całej rozmowy, a potem odkłada na półkę. To podejście zastosujemy w tej lekcji.

2.  **Przez Sub-Agenta (Delegacja)** – Główny agent powołuje do życia mniejszego agenta (tzw. SAP-Agenta), daje mu umiejętność i mówi: "zrób to i wróć z wynikiem". Taki sub-agent ma "czystą głowę" (brak historii rozmowy), co jest idealne przy zadaniach wymagających precyzji, by uniknąć szumu informacyjnego.

To prowadzi nas do kluczowego rozróżnienia inżynierskiego.

## Rozróżnienie: *Skill* vs *Sub-Agent*

Gdzie przebiega granica? Kiedy *Skill* staje się *Agentem*?

### 1. Sub-Agenty (podejście `Active`)
Jeżeli klasa ma metodę `execute()` i **wykonuje API call**, to technicznie nie jest już "umiejętnością" – jest **Agentem** (lub Sub-Agentem).

```python
class PointASubAgent:
    def __init__(self, llm_client):
        self.client = llm_client  # Posiada klienta LLM

    def execute(self, message, state):
        # ... buduje prompt ...
        response = self.client.call(prompt)  # <--- ROBI API CALL
        return response
```
Taki byt jest "autonomiczny". Możesz go poprosić o wykonanie zadania, a on sam martwi się o komunikację z modelem. To świetne podejście przy bardzo złożonych systemach (Swarms), ale trudniejsze w monitorowaniu i testowaniu (każdy sub-agent to "czarna skrzynka").



#### Charakterystyka:
- SubAgent jest **autonomicznym agentem** (w sensie, że ma dostęp do LLM client i może zrobić API Call)
- SubAgent **wykonuje** zadanie (execute method robi LLM call)
- Agent **deleguje** (przekazuje kontrolę do sub-agenta)
- Każdy SubAgent może mieć **własną strategię** (retry logic, cost tracking)

#### Zalety:
- ✅ SubAgent ma swoją autonomię (w stosunku do agenta głównego)
- ✅ Elastyczność (kontroluje swoje wywołania)
- ✅ Podobieństwo do ludzkiej organizacji (delegacja zadań)

#### Wady:
- ❌ Rozproszona kontrola (LLM calls w wielu miejscach)
- ❌ Trudniejszy monitoring (jak śledzić koszty/logi?)
- ❌ Complexity: trzeba zarządzać LLM klientem
- ❌ Testing: trzeba mockować LLM w każdym skillu



### 2. Czyste umiejętności (podejście `Passive`) 
To podejście, które wdrożymy teraz. Skill jest tylko **deklaracją**.

```python
class PointASkill:
    """Skill DEKLARUJE, co chce zrobić (nie WYKONUJE)"""

    def get_prompt_template(self):
        return Template("You are collecting: {{fields}}")

    @property
    def output_model(self):
        return PointAOutput

    # BRAK execute() - skill nie wykonuje, tylko dostarcza dane!
```

#### Charakterystyka:
- Skill jest **deklaracją** (template + schema)
- Skill **NIE WYKONUJE** (brak execute(), brak LLM calls)
- Agent **wykonuje** (single point of execution)
- Skill jest **pasywny** (agent go używa)


#### Zalety:
- ✅ **Centralized control**: Wszystkie wywołania API (koszty, logi, błędy) są w jednym miejscu (np. TutorAgent)
- ✅ **Łatwy monitoring**: jeden punkt do logowania/kosztów
- ✅ **Prostsze testing**: mock LLM tylko w jednym miejscu
- ✅ **Jasna architektura**: Agent = executor, Skill = declaration

#### Wady:
- ❌ Mniej autonomii (*skills* są zależne od agenta)
- ❌ Agent musi "wiedzieć", jak wykonać *skill* (to zresztą trudny problem, ale i tak pojawia się niezależnie od tego, jak dekomponujemy rozwiązanie [używamy *skill* czy nie] — tutaj trzeba nauczyć Agenta AI sprytnie wybierać, co w danej chwili chcemy osiągnąć).

W tej lekcji budujemy system w oparciu o **Podejście 2**. Nasz Agent będzie inteligentnym wykonawcą, który dynamicznie dobiera pasywne umiejętności z biblioteki.

---


### ☝️ Zwróć uwagę

Nie chciałbym, żeby teraz pojawiło się wrażenie, że stawiamy tu sprawę na ostrzu noża w stylu "Skillsy kontra Sub-Agenci". To nie jest konkurs, w którym jedno podejście wyklucza drugie.

Sub-Agenci to również sensowna koncepcja, która w wielu zastosowaniach robi niesamowitą robotę — na przykład wtedy, gdy musimy zrównoleglić pracę. Wyobraź sobie dziesięciu agentów przeszukujących różne źródła jednocześnie. To daje ogromną prędkość, ale też drastycznie podnosi poprzeczkę trudności: debugowanie procesów równoległych, które psują się w nieoczekiwanych momentach, to inżynierskie wyzwanie zupełnie innej klasy.

To, co robimy tutaj, to budowanie solidnych fundamentów. Pamiętaj, że Sub-Agent też może (i raczej powinien) korzystać ze *Skills*. Jeśli jednak rzucimy się na głęboką wodę architektury bez zrozumienia, jak modułowo budować proste umiejętności, utoniemy w chaosie.

Dlatego na ten moment skupiamy się na czystej definicji *Skill*. Traktuj to jako niezbędny krok, by w przyszłości wejść na poziom "wyższej szkoły jazdy", gdzie autonomiczni agenci będą żonglować tymi umiejętnościami samodzielnie.

## Warstwa 1 i 2: Koordynacja i egzekucja 

Skoro mamy już teorię za sobą, zejdźmy na ziemię. Zobaczmy, jak wygląda szkielet systemu, w który za chwilę będziesz wpinać swoje nowe umiejętności. Poznajmy jeszcze "Koordynatora" :).

Pamiętasz analogię z restauracją? Przyjrzyjmy się teraz zapleczu.

### AgentCoordinator (mózg operacyjny)

To tutaj zapada kluczowa decyzja: **"Co robimy dalej?"**.

Wielu programistów popełnia błąd, pytając LLM-a: *"Użytkownik napisał X, co mam teraz zrobić?"*. To jest:
1.  **Drogie** (zbędne tokeny).
2.  **Wolne** (latency).
3.  **Nieprzewidywalne** (model może wymyślić coś dziwnego).

My robimy inaczej. Używamy **twardej logiki** (State Machine). Jeśli proces onboardingu nie jest zakończony, to nie pytamy modelu o zdanie – po prostu uruchamiamy skill onboardingowy. To jest deterministyczne i bezpieczne.

Zobacz fragment kodu `AgentCoordinator`:

```python
# engine/agent_coordinator.py

def decide_next_skill(self, state: dict) -> str:
    """
    Decyzja o routingu oparta na twardej logice biznesowej.
    Tu nie ma AI - tu jest maszyna stanów.
    """
    workflow = state["workflow"]["onboarding_progress"]

    # 1. Priorytet: Dokończ Onboarding (Point A)
    if workflow["point_a_in_progress"] or not workflow["point_a_done"]:
        return "point_a"

    # 2. Priorytet: Ustal Cele (Point B)
    # (To odkomentujemy później, gdy stworzysz ten skill)
    # if workflow["point_b_in_progress"] or not workflow["point_b_done"]:
    #     return "point_b"

    # 3. Domyślnie: Tryb Nauczyciela
    return "tutoring"
```

Widzisz? Zero magii. Czysty `if/else`. To gwarantuje, że student nie przeskoczy do nauki, dopóki nie przejdzie oceny poziomu.

### TutorAgent (wykonawca)

Skoro Coordinator wybrał *skill* (np. `"point_a"`), przekazuje pałeczkę do `TutorAgent`.

To jest **jedyne miejsce w systemie**, które dotyka zewnętrznego API. To nasz "Single Point of Control". Dlaczego to takie ważne? Wiele powodów. Na przykład, gdy będziesz chcieć dodać obsługę błędów (retry), liczniki kosztów albo zmienić dostawcę modelu, zrobisz to w tym jednym pliku.

```python
# engine/tutor_agent.py

def execute(self, skill_name: str, user_message: str, state: dict) -> dict:
    # 1. Pobierz definicję skilla (przepis)
    skill = self._get_skill(skill_name)

    # 2. Przygotuj kontekst i wyrenderuj prompt (Jinja2)
    # Zauważ: Skill tylko PRZYGOTOWUJE treść, nie wysyła jej!
    context = self._prepare_context(user_message, state, skill_name)
    prompt = skill.prepare_prompt(context)

    # 3. Wykonaj połączenie (API Call)
    # ⚠️ To jedyne miejsce, gdzie wydajemy pieniądze ⚠️
    response = self.client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_message}
        ],
        response_model=skill.output_model  # Structured Output (Pydantic)
    )

    return {"ai_response": response, "new_state": state}
```

## Implementacja Skills (Prototypowanie w Notebooku) 

Teraz czas na Ciebie. Masz system, masz silnik, ale jest on "pusty". Musisz napisać przepisy (Skills).

Zamiast od razu tworzyć pliki w projekcie, zrobimy to **tutaj, w Notebooku**. To pozwoli nam na szybką pętlę zwrotną (Hot-Reload w głowie). Dopiero gdy upewnimy się, że skill działa, wdrożysz go do repozytorium.

Każdy Skill składa się z 3 elementów. To jest Twój standardowy "boilerplate":

1.  **Schema (Pydantic):** Co chcemy wyciągnąć? (Struktura danych)
2.  **Template (Jinja2):** Jaka jest instrukcja? (Prompt systemowy)
3.  **Class (Python):** Sklejenie powyższych. (Deklaracja)


## Wczytujemy biblioteki

Aby wszystko działało, musimy zaimportować niezbędne narzędzia. Używamy `pydantic` do strukturyzowania danych i `jinja2` do templatingu promptów.

In [ ]:
from abc import ABC, abstractmethod
from pydantic import BaseModel, Field
from jinja2 import Template
from typing import Optional

### Definicja klasy bazowej: `Skill`

Zanim przejdziesz do zadań, zdefiniujmy fundament. W Software 3.0, aby uniknąć chaosu, każda umiejętność musi spełniać ten sam **kontrakt programistyczny**.

Tworzymy klasę abstrakcyjną `Skill`, która wymusza na programiście (czyli na Tobie) zaimplementowanie trzech kluczowych elementów:
1.  **`name`**: Unikalny identyfikator (np. "greeting"). Po tym kluczu Agent będzie wybierał tę umiejętność.
2.  **`output_model`**: Schemat danych (Pydantic). Mówimy modelowi: "Nie zwracaj mi tekstu, zwróć mi wypełniony formularz".
3.  **`get_prompt_template`**: Metoda zwracająca obiekt `Template` (Jinja2). To jest Twój prompt systemowy, ale dynamiczny.

Uruchom poniższą komórkę, aby zarejestrować klasę bazową:

In [ ]:
class Skill(ABC):
    @property
    @abstractmethod
    def name(self) -> str:
        pass
        
    @property
    @abstractmethod
    def output_model(self) -> type[BaseModel]:
        pass

    @abstractmethod
    def get_prompt_template(self) -> Template:
        pass

## 🧠 Zadanie 2.5.1: Pierwsza umiejętność (Hello World)

Zacznijmy od podstaw. Twoim celem jest stworzenie umiejętności powitania (`SimpleGreetingSkill`).

Dlaczego to robimy? Musisz "poczuć", jak oddzielić **definicję** umiejętności (klasę) od jej **wykonania**. Tworzysz tutaj "kartridż", który później włożymy do konsoli (Agenta).

### Twoje zadanie:

W nowej komórce kodu poniżej zaimplementuj kod składający się z dwóch części:

1.  **Schema (`SimpleGreetingOutput`)**: Zdefiniuj model danych, który wymusi na LLM zwrócenie trzech konkretnych informacji:
    *   `ai_response` (str): Treść powitania dla użytkownika.
    *   `detected_language` (str): Kod języka (np. 'PL', 'EN'), w którym napisał użytkownik.
    *   `user_mood` (str): Wykryty nastrój użytkownika (np. "happy", "frustrated", "curious").

2.  **Skill (`SimpleGreetingSkill`)**: Zaimplementuj klasę dziedziczącą po `Skill`.
    *   W metodzie `get_prompt_template` stwórz prompt, który wciela się w rolę (korzystając ze zmiennych `{{ core.name }}` i `{{ core.role }}`) i realizuje powyższe zadanie.

> **💡 Wskazówka:** Pamiętaj, że metoda `get_prompt_template` musi zwracać obiekt `Template(...)`, a nie zwykły string. Dzięki temu Agent będzie mógł później bezpiecznie wstrzyknąć tam dane.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

Struktura powinna wyglądać tak. Zwróć uwagę na brak metody `execute`! Ten kod jest całkowicie pasywny.

```python
class SimpleGreetingSkill(Skill):
    @property
    def name(self) -> str:
        return "simple_greeting"
    
    # ... tu zdefiniuj output_model ...

    def get_prompt_template(self) -> Template:
        # Zwracamy obiekt Template, nie gotowy string!
        # Używamy zmiennych {{ core.name }} itd.
        return Template("""
You are {{ core.name }}...
TASK: Greet the user...
""")
```
</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>

```python

# 1. Definiujemy strukturę wyjścia (Schema)
class SimpleGreetingOutput(BaseModel):
    ai_response: str = Field(description="Warm, friendly greeting message")
    detected_language: str = Field(description="ISO 639-1 code (e.g., 'pl', 'en')")
    # Zgodnie z instrukcją: wykrywamy nastrój
    user_mood: str = Field(description="Detected user mood (e.g., happy, frustrated, curious)")

# 2. Definiujemy Skill (Deklaracja)
class SimpleGreetingSkill(Skill):
    @property
    def name(self) -> str:
        return "simple_greeting"

    @property
    def output_model(self) -> type[BaseModel]:
        return SimpleGreetingOutput

    def get_prompt_template(self) -> Template:
        # 3. Definiujemy Prompt (Template)
        return Template("""
You are {{ core.name }}, a {{ core.role }}.

TASK: Greet the new user warmly.

INSTRUCTIONS:
1. Introduce yourself briefly.
2. Detect the language from the user's message.
3. Detect the user's mood (happy, sad, curious, etc.).
4. Keep the tone friendly and encouraging.

USER MESSAGE: {{ user_message }}
""")
```

</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

### Sprawdźmy, czy działa (Unit Test)

Uruchom poniższy kod. Jeśli poprawnie zdefiniowałeś klasę, zobaczysz wyrenderowany prompt z podstawionymi danymi ("Alice", "AI Tutor"). To symulacja tego, co dzieje się wewnątrz Agenta tuż przed wysłaniem zapytania do OpenAI.


### Przykładowy wynik:
```text
Skill Name: simple_greeting
Output Model: SimpleGreetingOutput
--------------------

You are Alice, a AI Tutor.

TASK: Greet the new user warmly.

INSTRUCTIONS:
1. Introduce yourself briefly.
2. Detect the language from the user's message.
3. Check if they seem ready to start the lesson immediately.
4. Keep the tone friendly and encouraging.

USER MESSAGE: Cześć! Chcę się uczyć angielskiego.

```

In [ ]:
skill = SimpleGreetingSkill()

# Symulacja kontekstu (to normalnie robi Agent)
dummy_context = {
    "core": {"name": "Alice", "role": "AI Tutor"},
    "user_message": "Cześć! Chcę się uczyć angielskiego."
}

# Renderowanie
rendered = skill.get_prompt_template().render(**dummy_context)

print(f"Skill Name: {skill.name}")
print(f"Output Model: {skill.output_model.__name__}")
print("-" * 20)
print(rendered)

## 🧠 Zadanie 2.5.2: Change Request (Point A Extended)

Świetnie, masz statyczny skill. Teraz wejdźmy w prawdziwe życie inżyniera AI.

Masz działający proces Onboardingu (Point A), który zbiera podstawowe dane. Nagle przychodzi Klient (lub Product Owner) i mówi:
> "Słuchaj, musimy wiedzieć, kiedy student chce się uczyć. Dodaj pytanie o preferowaną porę dnia."

To zadanie pokaże Ci, jak **zmiana w wymaganiach biznesowych** propaguje się przez warstwy systemu: Schema -> Prompt -> Logika Stanu.

### Twoje zadanie:

Przeprowadź modyfikację w trzech krokach:

1.  **Schema (`PointAOutputExtended`)**: Dodaj nowe pole `preferred_study_time` (string, opcjonalne).
2.  **Skill (`PointASkillExtended`)**: Zaktualizuj prompt w metodzie `get_prompt_template`. Musisz poinstruować model: "Sprawdź, czy masz już `preferred_study_time`. Jeśli nie – zapytaj o to".
3.  **Logika Stanu (`simulate_memory_update`)**: To najważniejsza część. Napisz funkcję, która przyjmuje `state` i `output`, zapisuje dane i **decyduje o zakończeniu procesu**. Proces jest skończony (`point_a_done = True`) TYLKO wtedy, gdy mamy wszystkie wymagane pola (również to nowe!).

> **⚠️ Uwaga:** Najczęstszy błąd? Zmieniamy prompt, model pyta o godzinę, użytkownik odpowiada, ale... my zapominamy zaktualizować warunek `if` w Pythonie, który sprawdza, czy "już mamy wszystko". W efekcie agent albo kończy za wcześnie, albo pętli się w nieskończoność.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

Twój kod powinien składać się z dwóch części: definicji Skilla i funkcji aktualizującej stan.

W prompcie (Jinja2) musisz poinstruować model, aby sprawdzał, co już zostało zebrane.
W funkcji Pythonowej (symulującej MemoryManager) musisz zaktualizować listę `required_fields`.

```python
# 1. Schema
class PointAOutputExtended(BaseModel):
    # ... skopiuj 7 pól z przykładu wyżej (jeśli były) lub stwórz skrócone ...
    preferred_study_time: str | None = Field(description="...")

# 2. Logic simulation
def simulate_memory_update(state, output):
    # ... zapisz dane do state ...
    
    # WARUNEK STOPU
    required_fields = [
        "name", "target_lang", ..., 
        "preferred_study_time" # <--- To jest klucz!
    ]
    # Sprawdź czy wszystkie są w state['collected']
```
</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>

```python


# 1. SCHEMA (Rozszerzona)
class PointAOutputExtended(BaseModel):
    # Dla uproszczenia w tym zadaniu używamy mniej pól, ale zasada ta sama
    name: Optional[str] = Field(None, description="Student's name")
    target_lang: Optional[str] = Field(None, description="Target language")
    # ... (inne pola)
    preferred_study_time: Optional[str] = Field(None, description="Preferred time of day for studying") # <--- NOWE
    ai_response: str = Field(description="Response to user")

# 2. SKILL 
class PointASkillExtended(Skill):
    @property
    def name(self) -> str:
        return "point_a_extended"

    @property
    def output_model(self) -> type[BaseModel]:
        return PointAOutputExtended

    def get_prompt_template(self) -> Template:
        return Template("""
You are {{ core.name }}. Collect missing info.

REQUIRED: name, target_lang, preferred_study_time.

ALREADY COLLECTED: {{ state.workflow.collected_fields }}

INSTRUCTIONS:
- If 'preferred_study_time' is missing, ask for it specifically!
- If all collected, thank the user.
""")

# 3. LOGIC (Symulacja Memory Managera)
def simulate_memory_update(state: dict, output: PointAOutputExtended) -> dict:
    """Ta funkcja udaje to, co robi MemoryManager w pliku .py"""
    
    collected = state["workflow"]["collected_fields"]
    
    # A. Zapisujemy dane (tylko jeśli model coś zwrócił)
    if output.name:
        state["semantic"]["profile"]["name"] = output.name
        if "name" not in collected: collected.append("name")
        
    if output.target_lang:
        state["semantic"]["profile"]["target_lang"] = output.target_lang
        if "target_lang" not in collected: collected.append("target_lang")
        
    # B. Zapisujemy NOWE pole
    if output.preferred_study_time:
        state["semantic"]["profile"]["preferred_study_time"] = output.preferred_study_time
        if "preferred_study_time" not in collected: collected.append("preferred_study_time")

    # C. Aktualizujemy definicję "DONE"
    # To jest moment krytyczny - musimy wiedzieć, że wymagamy teraz 3 pól, a nie 2!
    required_fields = ["name", "target_lang", "preferred_study_time"]
    
    is_done = all(field in collected for field in required_fields)
    state["workflow"]["point_a_done"] = is_done
    
    return state
```

</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## Weryfikacja

Sprawdźmy, czy Twój kod działa poprawnie. Poniższy test symuluje sytuację, w której mamy już imię i język, ale brakuje nam nowego pola (czasu nauki).

Jeśli Twoja logika (`required_fields`) jest poprawna, `point_a_done` powinno zmienić się na `True` dopiero PO otrzymaniu czasu.


### Oczekiwany wynik:
```
Collected fields: ['name', 'target_lang', 'preferred_study_time']
Is Point A Done? True
```

In [ ]:
# 1. Inicjalizacja stanu (Symulacja bazy danych)
state = {
    "core": {"name": "Tutor"},
    "semantic": {"profile": {}},
    "workflow": {
        "collected_fields": ["name", "target_lang"], # Mamy już imię i język!
        "point_a_done": False
    }
}

# 2. Symulacja odpowiedzi modelu (Model "zrozumiał" i podał czas)
# Załóżmy, że użytkownik napisał: "Wolę uczyć się wieczorami"
mock_output = PointAOutputExtended(
    name=None, # Już mamy
    target_lang=None, # Już mamy
    preferred_study_time="evening", # <--- Nowa informacja
    ai_response="Great, evening classes it is!"
)

# 3. Uruchomienie Twojej logiki aktualizacji
new_state = simulate_memory_update(state, mock_output)

# 4. Sprawdzenie
print(f"Collected fields: {new_state['workflow']['collected_fields']}")
print(f"Is Point A Done? {new_state['workflow']['point_a_done']}")

## 🧠 Zadanie 2.5.3: Integration (Point B - cel)

Zróbmy krok dalej. W poprzednim zadaniu tylko modyfikowaliśmy. Teraz zbudujesz kompletny moduł "Goal Setting" (Point B), który **korzysta z wiedzy zdobytej w Point A**.

To zadanie uczy **context injection**. Twój prompt nie powinien być generyczny ("Cześć, jaki jest Twój cel?"). Powinien używać imienia studenta, które zdobyliśmy wcześniej, aby brzmieć personalnie ("Cześć Marku, jaki jest Twój cel?").

### Wymagania:

Stwórz Schema, Skill i Logikę dla procesu zbierania celów (`PointBSkill`).
1.  **Dane do zebrania:** `goal_type` (np. praca, podróże) i `specific_goal` (np. awans, wyjazd do Hiszpanii).
2.  **Prompt:** W szablonie Jinja2 użyj zmiennej `{{ state.semantic.profile.name }}`, aby zwrócić się do użytkownika po imieniu. To symuluje wykorzystanie pamięci semantycznej.
3.  **Logika:** Zapisz zebrane dane do `state["semantic"]["goals"]`.

To jest Twój egzamin końcowy z tej lekcji. Musisz połączyć kropki: definicja skilla + użycie pamięci w prompcie + aktualizacja stanu.


### Twoje zadanie

W komórce poniżej zaimplementuj cały stack: `Schema`, `Skill` oraz funkcję logiki.

> **Wskazówka:** W szablonie promptu (Jinja2) postaraj się wykorzystać fakt, że znamy już imię użytkownika ze stanu (`{{ state.semantic.profile.name }}`). To sprawi, że pytanie o cele będzie brzmiało bardziej osobiście.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

Schemat jest podobny do Point A. Kluczowa różnica to struktura stanu. Zapisujemy dane do `state["semantic"]["goals"]`, a nie `profile`.

```python
# 1. SCHEMA
class PointBOutput(BaseModel):
    goal_type: Optional[str] = Field(None, description="Reason for learning: work, travel, exam...")
    specific_goal: Optional[str] = Field(None, description="Specific target e.g. 'IELTS 7.0'")
    preferred_materials: Optional[str] = Field(None, description="Preferred content type")
    ai_response: str = Field(...)

# 2. LOGIC
def update_goals_logic(state, output):
    # Utwórz słownik goals jeśli nie istnieje
    if "goals" not in state["semantic"]:
        state["semantic"]["goals"] = {}
        
    # Zapisz pola...
    # Zaktualizuj collected_fields_point_b...
    # Sprawdź czy mamy komplet 3 pól...
```
</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>

```python

# 1. SCHEMA
class PointBOutput(BaseModel):
    goal_type: Optional[str] = Field(None, description="Main goal: work, travel, hobby, exam")
    specific_goal: Optional[str] = Field(None, description="Specific target details")
    preferred_materials: Optional[str] = Field(None, description="Preferred learning materials")
    ai_response: str = Field(description="Chat response")

# 2. SKILL
class PointBSkill(Skill):
    @property
    def name(self) -> str:
        return "point_b"

    @property
    def output_model(self) -> type[BaseModel]:
        return PointBOutput

    def get_prompt_template(self) -> Template:
        return Template("""
You are {{ core.name }}.
User: {{ state.semantic.profile.name }} (Level: {{ state.semantic.profile.level }}).

TASK: Define learning goals.
REQUIRED: goal_type, specific_goal, preferred_materials.

COLLECTED: {{ state.workflow.goals_collected }}

INSTRUCTIONS:
1. Ask about missing fields naturally.
2. If user says "I want to travel", ask where and when (specific_goal).
3. Ask what materials they like (videos, text, audio).
""")

# 3. LOGIC (Symulacja Managera)
def update_goals_logic(state: dict, output: PointBOutput) -> dict:
    # Inicjalizacja sekcji w stanie, jeśli nie istnieje
    if "goals" not in state["semantic"]:
        state["semantic"]["goals"] = {}
    if "goals_collected" not in state["workflow"]:
        state["workflow"]["goals_collected"] = []

    collected = state["workflow"]["goals_collected"]

    # Aktualizacja pól
    if output.goal_type:
        state["semantic"]["goals"]["goal_type"] = output.goal_type
        if "goal_type" not in collected: collected.append("goal_type")

    if output.specific_goal:
        state["semantic"]["goals"]["specific_goal"] = output.specific_goal
        if "specific_goal" not in collected: collected.append("specific_goal")

    if output.preferred_materials:
        state["semantic"]["goals"]["preferred_materials"] = output.preferred_materials
        if "preferred_materials" not in collected: collected.append("preferred_materials")

    # Sprawdzenie ukończenia
    required = ["goal_type", "specific_goal", "preferred_materials"]
    state["workflow"]["point_b_done"] = all(f in collected for f in required)

    return state
```

</p>
</details>
</details>

In [ ]:
## YOUR CODE HERE

## 🐞 Zadanie 2.5.4*: Debugging (Detektyw)

Na koniec coś, co spotka Cię w pracy częściej niż pisanie nowego kodu: **szukanie błędu**.

Załóżmy, że możesz otrzymać zgłoszenie od QA:
> "Hej, agent pyta o preferowane materiały (videos/books), ja odpowiadam, agent potwierdza, że zrozumiał... ale w bazie danych to pole nadal jest puste! Przy kolejnym uruchomieniu pyta o to samo. Chyba ma amnezję."

To klasyczny przypadek **desynchronizacji**.
⚠️ **Ważne:** W tym scenariuszu **Model (LLM) działa w 100% poprawnie**! Zrozumiał użytkownika i wygenerował poprawny JSON. Błąd leży wyłącznie w warstwie logiki (Python), która zgubiła te dane po drodze.

### Twoje zadanie:

Poniżej znajduje się funkcja `buggy_update_logic` zawierająca błąd.
1.  Uruchom komórkę testową, aby zobaczyć **AssertionError**. To potwierdzenie istnienia błędu.
2.  Przeanalizuj kod i znajdź miejsce, w którym dane przepadają.
3.  Napisz poprawioną wersję funkcji `fixed_update_logic`, która sprawi, że test zaświeci się na zielono.

#### ❌ Kod z błędem:

In [ ]:
def buggy_update_logic(state: dict, output: PointBOutput) -> dict:
    collected = state["workflow"].setdefault("goals_collected", [])
    goals = state["semantic"].setdefault("goals", {})

    # Logika aktualizacji
    if output.goal_type:
        goals["goal_type"] = output.goal_type
        if "goal_type" not in collected: collected.append("goal_type")

    if output.specific_goal:
        goals["specific_goal"] = output.specific_goal
        if "specific_goal" not in collected: collected.append("specific_goal")

    # ... tutaj ktoś chyba poszedł na kawę w połowie pisania i zapomniał o trzecim polu ...

    return state


#### Test case (uruchom to, by zobaczyć błąd):

In [ ]:
# 1. Setup
test_state = {"semantic": {"goals": {}}, "workflow": {}}

# 2. Symulacja outputu z LLM
# ⚠️ UWAGA: Model zadziałał POPRAWNIE! Zwrócił pole 'preferred_materials'.
# Błąd nie leży w prompcie ani w modelu, tylko w tym, co robimy z tym obiektem dalej.
mock_output = PointBOutput(
    goal_type="work",
    specific_goal=None,
    preferred_materials="Podcasts and Netflix", # <--- TO JEST W WYNIKU!
    ai_response="Great!"
)

# 3. Próba aktualizacji za pomocą błędnej funkcji
try:
    new_state = buggy_update_logic(test_state, mock_output)
    
    # 4. Sprawdzenie czy Python zapisał to, co zwrócił Model
    saved_materials = new_state["semantic"]["goals"].get("preferred_materials")
    
    assert saved_materials == "Podcasts and Netflix", \
        f"BŁĄD KRYTYCZNY! Model zwrócił materiały, ale w stanie ich nie ma. Wartość w stanie: {saved_materials}"
        
    print("✅ Test przeszedł! (Jeśli widzisz ten napis w 'buggy' wersji, to coś jest nie tak z testem)")

except AssertionError as e:
    print(f"❌ ZŁAPANO BŁĄD LOGIKI: {e}")
    print("👉 Twoje zadanie: Spraw, aby Python nie gubił pola 'preferred_materials'.")


#### ✅ I tutaj wchodzi Twoje rozwiązanie:

W komórce poniżej napisz funkcję `fixed_update_logic`, która naprawia ten błąd.

<b />
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie zagadki 👈
    </summary>
<p>


Błąd polegał na braku bloku `if output.preferred_materials: ...`. LLM zwracał dane, ale funkcja ich nie przetwarzała.

    
```python
def fixed_update_logic(state: dict, output: PointBOutput) -> dict:
    collected = state["workflow"].setdefault("goals_collected", [])
    goals = state["semantic"].setdefault("goals", {})

    if output.goal_type:
        goals["goal_type"] = output.goal_type
        if "goal_type" not in collected: collected.append("goal_type")

    if output.specific_goal:
        goals["specific_goal"] = output.specific_goal
        if "specific_goal" not in collected: collected.append("specific_goal")

    # ✅ NAPRAWIONE: Dodano obsługę brakującego pola
    if output.preferred_materials:
        goals["preferred_materials"] = output.preferred_materials
        if "preferred_materials" not in collected: collected.append("preferred_materials")

    return state

# Weryfikacja
print("Uruchamiam test naprawczej funkcji...")
# ... (powtórz kod testowy używając fixed_update_logic)
```

</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🎁 Bonus: zobacz gotowy system w akcji (v0.3) 

W tej lekcji budowaliśmy poszczególne klocki. Ale wiem, że nic nie daje takiej satysfakcji, jak zobaczenie gotowej, działającej budowli. Dlatego, aby trochę przyspieszyć Twoją naukę, przygotowałem dla Ciebie coś specjalnego.

W folderze `module2/src` (który znajduje się na tym samym poziomie co katalog `module2/notebooks`) znajdziesz przykładową implementację tego systemu. 

To nie jest prosty skrypt. To projekt, który łączy w sobie wszystko, o czym mówiliśmy: koordynatora, głównego agenta, zarządcę pamięci oraz zestaw zdefiniowanych Umiejętności (*Skills*). Zrobiłem tam dla Ciebie kilka kroków do przodu, rozszerzając logikę, którą ćwiczyliśmy w notebooku.

To nie jest ukończony system, ale przynajmniej kilka kroków do przodu, aby dać Ci możliwość zobaczenia, dokąd to zmierza, i przetestowania go.

### 1. Uruchom i poczuj "flow"

Przede wszystkim – **pobaw się tym!** Nie musisz od razu analizować kodu. Uruchom projekt i zacznij rozmowę z Agentem.

Zaimplementowałem tam mechanizm, który pozwoli Ci obserwować "wnętrze mózgu" Agenta na żywo. Zwróć uwagę na:
- Jaki **Skill** jest w danej chwili aktywny?
- Jak zmienia się **Stan (State)** po Twojej odpowiedzi?
- Jak system płynnie przełącza się z trybu "Ankietera" (Point A) w tryb "Nauczyciela" (Tutoring).

To da Ci niesamowitą intuicję, jak te teoretyczne koncepcje przekładają się na realne doświadczenie użytkownika.


### Przewodnik po TechUI (Twoje centrum dowodzenia)

Aby ułatwić Ci eksplorację, przygotowałem prosty interfejs graficzny (TechUI). Nie jest to produkcyjny frontend, ale narzędzie diagnostyczne dla inżyniera AI. Oto co oznaczają poszczególne sekcje na ekranie:

1.  **Pole tekstowe:** Tutaj wpisujesz swoją wiadomość i wysyłasz ją do Agenta. Proste.
2.  **Twoja wiadomość:** Pojawia się bliżej prawej strony (standardowy UX czatu).
3.  **Odpowiedź Agenta + Debug:** Tu widzisz, co odpowiedział model. Ale zwróć uwagę na **mały trójkąt** przy wiadomości. To Twoje okno na świat logiki.
4.  **Tryb Inspekcji (Debug Mode):** Po rozwinięciu trójkąta widzisz metadane: ID sesji, nazwę aktywnego *Skilla*, liczbę wykonanych kroków. To dowód na to, że agent nie "zmyśla", ale wykonuje konkretny program.
5.  **Panel User ID:** Domyślnie ustawiony na `test_user`. Możesz to zmienić, aby symulować inną osobę. Pamiętaj: w naszym systemie **pamięć kręci się wokół użytkownika**. Zmieniając ID, zmieniasz kontekst – agent "zapomni" rozmowę z Janem i załaduje pamięć Anny.
6.  **Complete State (Na żywo!):** To najciekawsza rzecz do obserwowania. Ten panel aktualizuje się w czasie rzeczywistym. Patrz uważnie: gdy podasz imię, zobaczysz, jak `state.semantic.profile.name` wypełnia się na Twoich oczach. Gdy skończysz etap, zobaczysz jak flagi w `workflow` zmieniają się z `False` na `True`.
7.  **Eksport danych:** Opcja pobrania całej rozmowy wraz z metadanymi jako plik JSON. To niezwykle przydatne do późniejszej analizy lub tworzenia datasetów testowych (zwróć uwagę, że na razie dialog tworzymy manualnie, ale w Module 5 nauczymy się symulować takie rozmowy automatycznie).

![](../images/ai_tutor_v0.3.jpg)


> **Ważna uwaga na koniec:**
> To, co widzisz, to implementacja **MVP (Minimum Viable Product)**. Pewne rzeczy uprościłem celowo, abyś mógł szybko zobaczyć efekt "wow" bez tonięcia w tysiącach linii kodu. Miejscami system może zachowywać się sztywno – to normalne na tym etapie. Najważniejsze jest to, że architektura jest gotowa i otwarta. W każdej chwili możesz dodać kolejny punkt do stanu czy nowy *skill*, nie burząc fundamentów.


> **Uwaga techniczna:** Pliki w folderze `src` są ustawione jako *read-only*, abyś przez przypadek ich nie nadpisać. Traktuj to jako bezpieczną piaskownicę.





### Jak uruchomić?

W terminalu wpisz po prostu:

In [ ]:
!python ../src/app.py

Gdy skrypt powyżej będzie już działał ("mielił"), kliknij tutaj, aby otworzyć UI w nowej karcie:

## 👉👉 [Klik =>  TECH UI (Port 8080)](/hub/user-redirect/proxy/8080/) 👈👈


Przypominam, że Gradio blokuje działanie notebooka (zobaczysz `[*]` przy komórce z kodem). Aby przejść do dalszej części lekcji i zacząć pisać prawdziwy kod:
1.  Wróć do komórki powyżej.
2.  Naciśnij przycisk **Stop ⏹️** w menu Jupytera.
3.  Dopiero gdy `[*]` zniknie, przejdź dalej.


### Gdzie patrzeć, żeby nie oszaleć? (wskazówka dla ambitnych)

Kiedy zajrzysz do kodu, możesz poczuć się przytłoczony ilością plików. Spokojnie!

Wiele osób wpada w "czarną dziurę", próbując zrozumieć każdą linię kodu "sklejającego" (tzw. *glue code*), który łączy bazę danych z API. Mam dla Ciebie ważną radę: **to jest drugorzędne.** Ten kod integracyjny często jest powtarzalny, i nawet może być nawet generowany automatycznie.

To, co jest **kluczowe** i gdzie leży Twoja prawdziwa wartość jako Inżyniera AI, to folder:
📂 **`module2/src/skills/`** (i inne struktury), czyli tam gdzie są tematy konceptualnie. To się liczy!

Tam znajdziesz definicje. Tam decydujesz:
1.  **CO** agent ma zrobić (Prompt w Jinja2).
2.  **JAK** ma zwrócić dane (Schema w Pydantic).

Jeśli masz spędzić czas na analizie kodu, spędź go właśnie tam. Zobacz, jak precyzyjnie zdefiniowane są pola w `description` i jak skonstruowane są instrukcje w szablonach. To tam dzieje się magia. Reszta to tylko hydraulika.

Pamiętaj też, że ten projekt w `src` to **wersja demonstracyjna (MVP)**. Pokazuje kierunek, w którym system może się rozwijać. To dowód na to, że koncepcje, o których mówimy, "spinają się" w całość. Nie jest to jeszcze finalny produkt komercyjny, ale solidna baza, z której możesz czerpać inspirację do własnych rozwiązań.

Uruchom, przetestuj, zainspiruj się!

## Podsumowanie

Właśnie dotarliśmy do końca Modułu 2. Przeszliśmy wspólnie przez pięć intensywnych lekcji, budując warstwa po warstwie architekturę pamięci w paradygmacie Software 3.0.

Spójrzmy na fundamenty, które położyliśmy:
- **🏛️ Constitutional Memory:** Ustaliliśmy zasady i granice bezpieczeństwa.
- **⚡ Working & Episodic Memory:** Nauczyliśmy system zarządzać bieżącą rozmową i historią.
- **🎯 Workflow Memory:** Wprowadziliśmy maszynę stanów, by Agent nie gubił się w procesie.
- **🧠 Semantic Memory:** Zbudowaliśmy bazę wiedzy o użytkowniku.
- **🛠️ Procedural Memory (w tej lekcji):** Zdefiniowaliśmy, czym są umiejętności (*Skills*).

Bądźmy jednak szczerzy: to, że przeszliśmy przez te tematy, nie oznacza, że proces nauki jest zakończony. Każda z tych warstw to temat rzeka, który w produkcyjnych systemach dopieszcza się miesiącami.

### Dlaczego ten moduł jest tak krytyczny?
Dostęp do potężnych modeli językowych (LLM) ma dzisiaj każdy. To towar powszechny. Twoją **przewagą konkurencyjną** nie jest sam model, ale to, jak zbudujesz jego Pamięć. To właśnie unikalna struktura kontekstu, którą tu projektujemy, odróżnia wybitnego Agenta od zwykłego chatbota.

Dlatego gorąco zachęcam: nie traktuj tego modułu jako "odhaczonego". Wracaj do tych notatek. Eksperymentuj z tymi warstwami w swoich projektach. To tutaj leży klucz do budowania spersonalizowanych, inteligentnych systemów.

### Co dalej? Moduł 3 - nasz agent "wychodzi z piwnicy"

Mamy solidne fundamenty wewnętrzne ("Mózg"), ale pominęliśmy jedną warstwę z naszej mapy: **Resource Memory**.

Na ten moment nasz Agent żyje w zamknięciu – w symulacji tekstowej. Może zaplanować idealną podróż, ale nie potrafi sprawdzić ceny biletu w czasie rzeczywistym. W kolejnym module to zmienimy.

Skupimy się na narzędziach (*Actions/Tools*) i wyjściu na świat zewnętrzny. Agent zyska "ręce", by wykonywać akcje, łączyć się z API i faktycznie wpływać na rzeczywistość.

Do zobaczenia w Module 3!

## Przydatne linki

1. [Introducing Agent Skills](https://claude.com/blog/skills)
2. [Equipping agents for the real world with Agent Skills](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)
3. [Anthropic: Memory tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/memory-tool)
4. [Exploring Agent Procedural Memory](https://arxiv.org/pdf/2508.06433)
5. [Procedural Memory Is Not All You Need](https://arxiv.org/pdf/2505.03434)
6. [Procedural Knowledge In Pretraining Drives Reasoning In Large Language Models](https://arxiv.org/pdf/2411.12580)
7. [Making Sense of Memory in AI Agents](https://www.leoniemonigatti.com/blog/memory-in-ai-agents.html)